In [1]:
import pandas as pd
from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [2]:
DATA_PATH = Path("../data/processed/synthetic_feature_engineered.csv")

df = pd.read_csv(DATA_PATH)

df.head()

,age_months,sex,weight_kg,height_cm,muac_cm,waz,haz,whz,underweight_status,stunting_status,wasting_status,sam_muac_flag,mam_muac_flag,bmi,age_group,bmi_category
0,5,1,5.01,61.1,12.1,-2.13,-1.22,-1.73,moderate,normal,normal,False,True,13.420086,0-6,Low
1,46,1,15.21,103.3,14.7,-0.34,0.26,-0.42,normal,normal,normal,False,False,14.253731,36-48,Normal
2,39,0,15.13,100.7,15.2,0.34,0.86,0.06,normal,normal,normal,False,False,14.920384,36-48,Normal
3,26,0,4.60,78.0,10.6,-4.38,-2.36,-3.63,severe,moderate,severe,True,False,7.560815,24-36,Low
4,25,0,8.46,84.8,12.6,-1.98,-0.44,-1.84,normal,normal,normal,False,False,11.764640,24-36,Low


In [4]:
features = [
    "age_months",
    "sex",
    "weight_kg",
    "height_cm",
    "muac_cm",
    "waz",
    "haz",
    "whz",
    "bmi"
]

X = df[features]

In [5]:
targets = [
    "underweight_status",
    "stunting_status",
    "wasting_status"
]

In [6]:
model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

In [7]:
results = []

for target in targets:

    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    print(f"{target}: {accuracy:.4f}")

    joblib.dump(
        model,
        model_dir / f"{target}_model.pkl"
    )

    results.append({
        "Target": target,
        "Accuracy": accuracy
    })

underweight_status: 1.0000
stunting_status: 1.0000
wasting_status: 1.0000


In [8]:
results_df = pd.DataFrame(results)

results_df

,Target,Accuracy
0,underweight_status,1.0
1,stunting_status,1.0
2,wasting_status,1.0


In [9]:
results_df.to_csv(
    "../reports/multi_target_results.csv",
    index=False
)